# Notebook 06a: Ensemble Methods & Model Selection

**Purpose:** Build ensemble models, comprehensively evaluate all models across feature sets, and select final production model

**Key Objectives:**
1. Build ensemble models (Stacking, Voting) for each feature set
2. Compare baseline vs tuned vs ensemble across MFCC, GTCC, Combined
3. Perform error analysis (confusion matrices, FP/FN rates)
4. Measure production metrics (size, speed, memory)
5. Select final model for deployment

**Expected Results:**
- Ensemble improvement: +0.2-0.5% F1
- Final model: 98.5-99% F1 score
- Production ready: <50ms inference, <50MB size

---

## Section 1: Setup & Configuration

In [1]:
import os, sys
from pathlib import Path
import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib

# Sklearn
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, roc_auc_score
)

import matplotlib.pyplot as plt
import seaborn as sns

print('✓ All libraries imported successfully')

✓ All libraries imported successfully


In [2]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
FEATURES_DIR = PROJECT_ROOT / 'data' / 'features' / 'classical'
MODELS_DIR = PROJECT_ROOT / 'models' / 'classical'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'

# Create directories
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Models dir:   {MODELS_DIR}')
print(f'Results dir:  {RESULTS_DIR}')

Project root: /Users/harryirving/Development/projects/ai-ml/BikeAIv5
Models dir:   /Users/harryirving/Development/projects/ai-ml/BikeAIv5/models/classical
Results dir:  /Users/harryirving/Development/projects/ai-ml/BikeAIv5/results


In [3]:
# Configuration
FEATURE_SETS = ['mfcc', 'gtcc', 'combined']
MODEL_TYPES = ['XGBoost', 'Random Forest', 'LinearSVC']

print(f'\nAnalyzing {len(FEATURE_SETS)} feature sets: {FEATURE_SETS}')
print(f'With {len(MODEL_TYPES)} model types: {MODEL_TYPES}')
print(f'Total models to evaluate: {len(FEATURE_SETS) * len(MODEL_TYPES) * 2} (baseline + tuned)')
print(f'Plus {len(FEATURE_SETS) * 3} ensemble models (stacking, soft voting, hard voting)')


Analyzing 3 feature sets: ['mfcc', 'gtcc', 'combined']
With 3 model types: ['XGBoost', 'Random Forest', 'LinearSVC']
Total models to evaluate: 18 (baseline + tuned)
Plus 9 ensemble models (stacking, soft voting, hard voting)


In [4]:
# Quick fix for sample mismatch
print(f"Original shapes: X={X.shape}, y={y.shape}")

if X.shape[0] != y.shape[0]:
    print(f"\n⚠️ Sample count mismatch detected!")
    print(f"   X: {X.shape[0]} samples")
    print(f"   y: {y.shape[0]} samples")
    
    # Trim to smaller size
    min_samples = min(X.shape[0], y.shape[0])
    print(f"\n✂️ Trimming both to {min_samples} samples...")
    
    X = X[:min_samples]
    y = y[:min_samples]
    
    print(f"✓ Fixed shapes: X={X.shape}, y={y.shape}")
else:
    print("✓ Shapes match - no fix needed")


NameError: name 'X' is not defined

In [ ]:
import numpy as np
from pathlib import Path

FEATURES_DIR = Path('/Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/features/classical')

print("Checking actual file sizes...\n")

# Check 16kHz files
files_to_check = {
    'combined_features (16kHz)': 'combined_unbalanced_features.npy',
    'labels (16kHz)': 'labels.npy',
    'combined_features (44.1kHz)': 'combined_unbalanced_features_44khz.npy',
    'labels (44.1kHz)': 'labels_44khz.npy'
}

for name, filename in files_to_check.items():
    filepath = FEATURES_DIR / filename
    if filepath.exists():
        data = np.load(filepath)
        print(f"{name:30} {data.shape[0]:,} samples")
    else:
        print(f"{name:30} ❌ NOT FOUND")

print("\n" + "="*60)
print("DIAGNOSIS:")
print("="*60)

# Load the specific files notebook 06a uses
X = np.load(FEATURES_DIR / 'combined_unbalanced_features.npy')
y = np.load(FEATURES_DIR / 'labels.npy')

print(f"\nX shape: {X.shape} ({X.shape[0]:,} samples)")
print(f"y shape: {y.shape} ({y.shape[0]:,} samples)")

if X.shape[0] == y.shape[0]:
    print("\n✅ Files match! The error must be coming from elsewhere.")
else:
    print(f"\n❌ MISMATCH FOUND!")
    print(f"   Difference: {abs(X.shape[0] - y.shape[0])} samples")
    print(f"\n   This is the problem! Files are out of sync.")


Checking actual file sizes...

combined_features (16kHz)      59,869 samples
labels (16kHz)                 59,869 samples
combined_features (44.1kHz)    59,869 samples
labels (44.1kHz)               59,869 samples

DIAGNOSIS:

X shape: (59869, 248) (59,869 samples)
y shape: (59869,) (59,869 samples)

✅ Files match! The error must be coming from elsewhere.


## Section 2: Load All Models and Data

In [ ]:
print('Loading all trained models and data...')
print('='*70)

# Storage for loaded models and data
loaded_models = {}
test_data = {}
scalers = {}

for feature_set in FEATURE_SETS:
    print(f'\nFeature set: {feature_set.upper()}')
    
    # Load test data
    if feature_set == 'mfcc':
        feature_file = FEATURES_DIR / 'mfcc_unbalanced_features.npy'
    elif feature_set == 'gtcc':
        feature_file = FEATURES_DIR / 'gtcc_unbalanced_features.npy'
    else:  # combined
        feature_file = FEATURES_DIR / 'combined_unbalanced_features.npy'
    
    labels_file = FEATURES_DIR / 'labels.npy'
    
    X = np.load(feature_file)
    y = np.load(labels_file)
    
    # Recreate the same train/test split as in notebook 05
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.15, random_state=42, stratify=y
    )
    
    # Load scaler
    scaler_path = MODELS_DIR / f'scaler_{feature_set}.pkl'
    scaler = joblib.load(scaler_path)
    X_test_scaled = scaler.transform(X_test)
    
    test_data[feature_set] = {'X': X_test_scaled, 'y': y_test}
    scalers[feature_set] = scaler
    
    print(f'  Test samples: {X_test.shape[0]:,}')
    print(f'  Features: {X_test.shape[1]}')
    
    # Load models
    loaded_models[feature_set] = {}
    
    for model_type in MODEL_TYPES:
        # Try to load tuned model
        if model_type == 'XGBoost':
            model_path = MODELS_DIR / f'xgboost_tuned_{feature_set}.pkl'
        elif model_type == 'Random Forest':
            model_path = MODELS_DIR / f'random_forest_tuned_{feature_set}.pkl'
        else:  # LinearSVC
            model_path = MODELS_DIR / f'linearsvc_tuned_{feature_set}.pkl'
        
        if model_path.exists():
            loaded_models[feature_set][model_type] = joblib.load(model_path)
            print(f'  ✓ Loaded: {model_type}')
        else:
            print(f'  ⚠️  Missing: {model_type} (will skip)')

print('\n' + '='*70)
print('✓ All models and data loaded successfully')

Loading all trained models and data...

Feature set: MFCC


ValueError: Found input variables with inconsistent numbers of samples: [60324, 59869]

In [ ]:
print('\nChecking and handling NaN values...')
print('='*70)

for feature_set in FEATURE_SETS:
    if feature_set not in test_data:
        continue
    
    X_test = test_data[feature_set]['X']
    
    # Check for NaN
    nan_count = np.isnan(X_test).sum()
    
    if nan_count > 0:
        print(f'\n{feature_set.upper()}: Found {nan_count:,} NaN values')
        print(f'  Applying median imputation...')
        
        from sklearn.impute import SimpleImputer
        imputer = SimpleImputer(strategy='median')
        
        # Fit on test data and transform
        X_test_imputed = imputer.fit_transform(X_test)
        
        # Update test data
        test_data[feature_set]['X'] = X_test_imputed
        
        # Verify
        remaining_nans = np.isnan(X_test_imputed).sum()
        if remaining_nans == 0:
            print(f'  ✓ All NaN values handled')
        else:
            print(f'  ⚠️  Still have {remaining_nans} NaN values')
    else:
        print(f'\n{feature_set.upper()}: ✓ No NaN values detected')

print('\n' + '='*70)
print('✓ NaN handling complete for all feature sets')



Checking and handling NaN values...

MFCC: Found 2 NaN values
  Applying median imputation...
  ✓ All NaN values handled

GTCC: Found 2 NaN values
  Applying median imputation...
  ✓ All NaN values handled

COMBINED: Found 2 NaN values
  Applying median imputation...
  ✓ All NaN values handled

✓ NaN handling complete for all feature sets


## Section 3: Build Ensemble Models

### 3.1 Stacking Classifier

In [ ]:
print('\nBuilding STACKING CLASSIFIERS...')
print('='*70)

ensemble_models = {}

for feature_set in FEATURE_SETS:
    print(f'\nFeature set: {feature_set.upper()}')
    
    # Get available models for this feature set
    available_models = loaded_models[feature_set]
    
    if len(available_models) < 2:
        print(f'  ⚠️  Insufficient models ({len(available_models)}) - skipping ensemble')
        continue
    
    # Prepare base estimators
    base_estimators = [(name, model) for name, model in available_models.items()]
    
    print(f'  Base estimators: {[name for name, _ in base_estimators]}')
    
    # Build stacking classifier with Logistic Regression meta-learner
    stacking_clf = StackingClassifier(
        estimators=base_estimators,
        final_estimator=LogisticRegression(max_iter=1000, random_state=42),
        cv=5,
        n_jobs=-1
    )
    
    # Train on training data (need to recreate train set)
    print(f'  Training stacking classifier...')
    start_time = time.time()
    
    # Load and prepare training data
    if feature_set == 'mfcc':
        feature_file = FEATURES_DIR / 'mfcc_unbalanced_features.npy'
    elif feature_set == 'gtcc':
        feature_file = FEATURES_DIR / 'gtcc_unbalanced_features.npy'
    else:
        feature_file = FEATURES_DIR / 'combined_unbalanced_features.npy'
    
    X_full = np.load(feature_file)
    y_full = np.load(FEATURES_DIR / 'labels.npy')
    
    # Handle NaN values in training data
    nan_count = np.isnan(X_full).sum()
    if nan_count > 0:
        print(f'  ⚠️  Found {nan_count:,} NaN values in training data')
        print(f'  Applying median imputation...')
        from sklearn.impute import SimpleImputer
        imputer = SimpleImputer(strategy='median')
        X_full = imputer.fit_transform(X_full)
        remaining_nans = np.isnan(X_full).sum()
        print(f'  ✓ Handled - {remaining_nans} NaNs remaining')
        
    # Same split as notebook 05
    X_temp, X_test, y_temp, y_test = train_test_split(
        X_full, y_full, test_size=0.15, random_state=42, stratify=y_full
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp
    )
    
    # Scale
    X_train_scaled = scalers[feature_set].transform(X_train)
    
    # Fit stacking classifier
    stacking_clf.fit(X_train_scaled, y_train)
    
    train_time = time.time() - start_time
    
    # Store
    if feature_set not in ensemble_models:
        ensemble_models[feature_set] = {}
    ensemble_models[feature_set]['Stacking'] = stacking_clf
    
    print(f'  ✓ Stacking trained in {train_time:.1f}s')
    
    # Evaluate
    y_pred = stacking_clf.predict(test_data[feature_set]['X'])
    test_f1 = f1_score(test_data[feature_set]['y'], y_pred)
    test_acc = accuracy_score(test_data[feature_set]['y'], y_pred)
    
    print(f'  Test Accuracy: {test_acc:.4f}')
    print(f'  Test F1:       {test_f1:.4f}')
    
    # Save model
    model_path = MODELS_DIR / f'stacking_{feature_set}.pkl'
    joblib.dump(stacking_clf, model_path)
    print(f'  Saved: {model_path.name}')

print('\n' + '='*70)
print('✓ Stacking classifiers complete')


Building STACKING CLASSIFIERS...

Feature set: MFCC
  Base estimators: ['XGBoost', 'Random Forest', 'LinearSVC']
  Training stacking classifier...
  ⚠️  Found 20 NaN values in training data
  Applying median imputation...
  ✓ Handled - 0 NaNs remaining
  ✓ Stacking trained in 1433.0s
  Test Accuracy: 0.9842
  Test F1:       0.9856
  Saved: stacking_mfcc.pkl

Feature set: GTCC
  Base estimators: ['XGBoost', 'Random Forest', 'LinearSVC']
  Training stacking classifier...
  ⚠️  Found 20 NaN values in training data
  Applying median imputation...
  ✓ Handled - 0 NaNs remaining
  ✓ Stacking trained in 216.4s
  Test Accuracy: 0.9812
  Test F1:       0.9829
  Saved: stacking_gtcc.pkl

Feature set: COMBINED
  Base estimators: ['XGBoost', 'Random Forest', 'LinearSVC']
  Training stacking classifier...
  ⚠️  Found 20 NaN values in training data
  Applying median imputation...
  ✓ Handled - 0 NaNs remaining
  ✓ Stacking trained in 154.7s
  Test Accuracy: 0.9861
  Test F1:       0.9873
  Saved: s

Blending using previous models

In [ ]:
print('\nBuilding STACKING CLASSIFIERS (using pre-trained models)...')
print('='*70)

ensemble_models = {}

for feature_set in FEATURE_SETS:
    print(f'\nFeature set: {feature_set.upper()}')
    
    # Get available models for this feature set
    available_models = loaded_models[feature_set]
    
    if len(available_models) < 2:
        print(f'  ⚠️  Insufficient models ({len(available_models)}) - skipping')
        continue
    
    print(f'  Using pre-trained models: {list(available_models.keys())}')
    
    # Get validation predictions from each model
    print(f'  Generating meta-features from validation set...')
    
    # Recreate validation set
    if feature_set == 'mfcc':
        feature_file = FEATURES_DIR / 'mfcc_unbalanced_features.npy'
    elif feature_set == 'gtcc':
        feature_file = FEATURES_DIR / 'gtcc_unbalanced_features.npy'
    else:
        feature_file = FEATURES_DIR / 'combined_unbalanced_features.npy'
    
    X_full = np.load(feature_file)
    y_full = np.load(FEATURES_DIR / 'labels.npy')
    
    # Handle NaN
    nan_count = np.isnan(X_full).sum()
    if nan_count > 0:
        from sklearn.impute import SimpleImputer
        imputer = SimpleImputer(strategy='median')
        X_full = imputer.fit_transform(X_full)
    
    # Same splits
    X_temp, X_test, y_temp, y_test = train_test_split(
        X_full, y_full, test_size=0.15, random_state=42, stratify=y_full
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp
    )
    
    # Scale
    X_val_scaled = scalers[feature_set].transform(X_val)
    X_test_scaled = test_data[feature_set]['X']
    
    # Get predictions from each base model on VALIDATION set
    meta_features_val = []
    meta_features_test = []
    
    for model_name, model in available_models.items():
        # Validation predictions (for training meta-learner)
        if hasattr(model, 'predict_proba'):
            val_pred_proba = model.predict_proba(X_val_scaled)
            meta_features_val.append(val_pred_proba)
            
            test_pred_proba = model.predict_proba(X_test_scaled)
            meta_features_test.append(test_pred_proba)
        else:
            # If no predict_proba, use predictions
            val_pred = model.predict(X_val_scaled).reshape(-1, 1)
            meta_features_val.append(val_pred)
            
            test_pred = model.predict(X_test_scaled).reshape(-1, 1)
            meta_features_test.append(test_pred)
    
    # Stack meta-features
    X_meta_val = np.hstack(meta_features_val)
    X_meta_test = np.hstack(meta_features_test)
    
    print(f'  Meta-features shape: {X_meta_val.shape}')
    
    # Train meta-learner on validation predictions
    print(f'  Training meta-learner...')
    start_time = time.time()
    
    meta_learner = LogisticRegression(max_iter=1000, random_state=42)
    meta_learner.fit(X_meta_val, y_val)
    
    train_time = time.time() - start_time
    
    print(f'  ✓ Meta-learner trained in {train_time:.1f}s')
    
    # Create custom stacking predictor
    class FastStackingClassifier:
        def __init__(self, base_models, meta_learner, scaler):
            self.base_models = base_models
            self.meta_learner = meta_learner
            self.scaler = scaler
        
        def predict(self, X):
            # Get predictions from base models
            meta_features = []
            for model in self.base_models.values():
                if hasattr(model, 'predict_proba'):
                    pred_proba = model.predict_proba(X)
                    meta_features.append(pred_proba)
                else:
                    pred = model.predict(X).reshape(-1, 1)
                    meta_features.append(pred)
            
            X_meta = np.hstack(meta_features)
            return self.meta_learner.predict(X_meta)
        
        def predict_proba(self, X):
            # Get predictions from base models
            meta_features = []
            for model in self.base_models.values():
                if hasattr(model, 'predict_proba'):
                    pred_proba = model.predict_proba(X)
                    meta_features.append(pred_proba)
                else:
                    pred = model.predict(X).reshape(-1, 1)
                    meta_features.append(pred)
            
            X_meta = np.hstack(meta_features)
            return self.meta_learner.predict_proba(X_meta)
    
    stacking_clf = FastStackingClassifier(available_models, meta_learner, scalers[feature_set])
    
    # Store
    if feature_set not in ensemble_models:
        ensemble_models[feature_set] = {}
    ensemble_models[feature_set]['Stacking'] = stacking_clf
    
    # Evaluate on test set
    y_pred = stacking_clf.predict(X_test_scaled)
    test_f1 = f1_score(test_data[feature_set]['y'], y_pred)
    test_acc = accuracy_score(test_data[feature_set]['y'], y_pred)
    
    print(f'  Test Accuracy: {test_acc:.4f}')
    print(f'  Test F1:       {test_f1:.4f}')
    
    # Save model
    model_path = MODELS_DIR / f'Blending_{feature_set}.pkl'
    joblib.dump(stacking_clf, model_path)
    print(f'  Saved: {model_path.name}')

print('\n' + '='*70)
print('✓ Stacking classifiers complete (using pre-trained models)')
print('='*70)



Building STACKING CLASSIFIERS (using pre-trained models)...

Feature set: MFCC
  Using pre-trained models: ['XGBoost', 'Random Forest', 'LinearSVC']
  Generating meta-features from validation set...
  Meta-features shape: (9028, 6)
  Training meta-learner...
  ✓ Meta-learner trained in 0.0s
  Test Accuracy: 0.9842
  Test F1:       0.9856
  Saved: stacking_mfcc.pkl

Feature set: GTCC
  Using pre-trained models: ['XGBoost', 'Random Forest', 'LinearSVC']
  Generating meta-features from validation set...
  Meta-features shape: (9028, 6)
  Training meta-learner...
  ✓ Meta-learner trained in 0.0s
  Test Accuracy: 0.9809
  Test F1:       0.9826
  Saved: stacking_gtcc.pkl

Feature set: COMBINED
  Using pre-trained models: ['XGBoost', 'Random Forest', 'LinearSVC']
  Generating meta-features from validation set...
  Meta-features shape: (9028, 6)
  Training meta-learner...
  ✓ Meta-learner trained in 0.0s
  Test Accuracy: 0.9863
  Test F1:       0.9875
  Saved: stacking_combined.pkl

✓ Stackin

### 3.2 Soft Voting Classifier

In [ ]:
print('\nBuilding SOFT VOTING CLASSIFIERS...')
print('='*70)

for feature_set in FEATURE_SETS:
    print(f'\nFeature set: {feature_set.upper()}')
    
    # Get available models
    available_models = loaded_models[feature_set]
    
    if len(available_models) < 2:
        print(f'  ⚠️  Insufficient models - skipping')
        continue
    
    # Create list of (name, model) tuples
    estimators = [(name, model) for name, model in available_models.items()]
    print(f'  Combining models: {[name for name, _ in estimators]}')
    
    # Create voting classifier
    voting_clf = VotingClassifier(
        estimators=estimators,
        voting='soft',  # Average probabilities
        n_jobs=-1
    )
    
    # **FIX: Must fit the voting classifier to set up label encoder**
    # Use a small subset of training data for this
    print(f'  Setting up voting classifier...')
    
    # Load a small sample of training data
    if feature_set == 'mfcc':
        feature_file = FEATURES_DIR / 'mfcc_unbalanced_features.npy'
    elif feature_set == 'gtcc':
        feature_file = FEATURES_DIR / 'gtcc_unbalanced_features.npy'
    else:
        feature_file = FEATURES_DIR / 'combined_unbalanced_features.npy'
    
    X_full = np.load(feature_file)
    y_full = np.load(FEATURES_DIR / 'labels.npy')
    
    # Handle NaN
    nan_count = np.isnan(X_full).sum()
    if nan_count > 0:
        from sklearn.impute import SimpleImputer
        imputer = SimpleImputer(strategy='median')
        X_full = imputer.fit_transform(X_full)
    
    # Use small sample just to fit the voting wrapper (won't retrain base models)
    X_sample, _, y_sample, _ = train_test_split(
        X_full, y_full, train_size=100, random_state=42, stratify=y_full
    )
    X_sample_scaled = scalers[feature_set].transform(X_sample)
    
    # Fit voting classifier (this just sets up the wrapper, doesn't retrain base models)
    voting_clf.fit(X_sample_scaled, y_sample)
    print(f'  ✓ Voting classifier ready')
    
    # Store
    if feature_set not in ensemble_models:
        ensemble_models[feature_set] = {}
    ensemble_models[feature_set]['Soft Voting'] = voting_clf
    
    # Evaluate
    y_pred = voting_clf.predict(test_data[feature_set]['X'])
    test_f1 = f1_score(test_data[feature_set]['y'], y_pred)
    test_acc = accuracy_score(test_data[feature_set]['y'], y_pred)
    
    print(f'  Test Accuracy: {test_acc:.4f}')
    print(f'  Test F1:       {test_f1:.4f}')
    
    # Save
    model_path = MODELS_DIR / f'voting_soft_{feature_set}.pkl'
    joblib.dump(voting_clf, model_path)
    print(f'  Saved: {model_path.name}')

print('\n' + '='*70)
print('✓ Soft voting classifiers complete')
print('='*70)



Building SOFT VOTING CLASSIFIERS...

Feature set: MFCC
  Combining models: ['XGBoost', 'Random Forest', 'LinearSVC']
  Setting up voting classifier...
  ✓ Voting classifier ready
  Test Accuracy: 0.8212
  Test F1:       0.8433
  Saved: voting_soft_mfcc.pkl

Feature set: GTCC
  Combining models: ['XGBoost', 'Random Forest', 'LinearSVC']
  Setting up voting classifier...
  ✓ Voting classifier ready
  Test Accuracy: 0.8087
  Test F1:       0.8294
  Saved: voting_soft_gtcc.pkl

Feature set: COMBINED
  Combining models: ['XGBoost', 'Random Forest', 'LinearSVC']
  Setting up voting classifier...
  ✓ Voting classifier ready
  Test Accuracy: 0.8409
  Test F1:       0.8581
  Saved: voting_soft_combined.pkl

✓ Soft voting classifiers complete


### 3.3 Hard Voting Classifier

In [ ]:
print('\nBuilding HARD VOTING CLASSIFIERS...')
print('='*70)

class ManualHardVoting:
    """Simple hard voting without sklearn wrapper"""
    def __init__(self, models):
        self.models = models
    
    def predict(self, X):
        # Get predictions from all models
        predictions = []
        for model in self.models.values():
            predictions.append(model.predict(X))
        
        # Stack predictions (each row is one sample's votes)
        votes = np.column_stack(predictions)
        
        # Majority vote for each sample
        from scipy import stats
        majority_vote, _ = stats.mode(votes, axis=1, keepdims=False)
        
        return majority_vote.ravel()
    
    def predict_proba(self, X):
        # For hard voting, we can approximate probabilities
        # by converting predictions to one-hot and averaging
        predictions = []
        for model in self.models.values():
            pred = model.predict(X)
            predictions.append(pred)
        
        votes = np.column_stack(predictions)
        
        # Calculate proportion of votes for each class
        n_samples = X.shape[0]
        probas = np.zeros((n_samples, 2))
        
        for i in range(n_samples):
            unique, counts = np.unique(votes[i], return_counts=True)
            for cls, count in zip(unique, counts):
                probas[i, int(cls)] = count / len(self.models)
        
        return probas

for feature_set in FEATURE_SETS:
    print(f'\nFeature set: {feature_set.upper()}')
    
    available_models = loaded_models[feature_set]
    
    if len(available_models) < 2:
        print(f'  ⚠️  Insufficient models - skipping')
        continue
    
    print(f'  Combining models: {list(available_models.keys())}')
    
    # Create manual voting classifier
    voting_clf = ManualHardVoting(available_models)
    
    # Store
    if feature_set not in ensemble_models:
        ensemble_models[feature_set] = {}
    ensemble_models[feature_set]['Hard Voting'] = voting_clf
    
    # Evaluate (instant)
    y_pred = voting_clf.predict(test_data[feature_set]['X'])
    test_f1 = f1_score(test_data[feature_set]['y'], y_pred)
    test_acc = accuracy_score(test_data[feature_set]['y'], y_pred)
    
    print(f'  Test Accuracy: {test_acc:.4f}')
    print(f'  Test F1:       {test_f1:.4f}')
    
    # Save
    model_path = MODELS_DIR / f'voting_hard_{feature_set}.pkl'
    joblib.dump(voting_clf, model_path)
    print(f'  Saved: {model_path.name}')

print('\n' + '='*70)
print('✓ Hard voting classifiers complete')
print('='*70)



Building HARD VOTING CLASSIFIERS...

Feature set: MFCC
  Combining models: ['XGBoost', 'Random Forest', 'LinearSVC']
  Test Accuracy: 0.9779
  Test F1:       0.9800
  Saved: voting_hard_mfcc.pkl

Feature set: GTCC
  Combining models: ['XGBoost', 'Random Forest', 'LinearSVC']
  Test Accuracy: 0.9756
  Test F1:       0.9780
  Saved: voting_hard_gtcc.pkl

Feature set: COMBINED
  Combining models: ['XGBoost', 'Random Forest', 'LinearSVC']
  Test Accuracy: 0.9812
  Test F1:       0.9830
  Saved: voting_hard_combined.pkl

✓ Hard voting classifiers complete


## Section 4: Comprehensive Evaluation Matrix

### 4.1 Evaluate All Individual Models

In [ ]:
print('\nEVALUATING ALL MODELS...')
print('='*70)

all_results = []

for feature_set in FEATURE_SETS:
    print(f'\nFeature set: {feature_set.upper()}')
    
    X_test = test_data[feature_set]['X']
    y_test = test_data[feature_set]['y']
    
    # Evaluate individual models
    for model_name, model in loaded_models[feature_set].items():
        print(f'  Evaluating {model_name}...')
        
        # Predictions
        y_pred = model.predict(X_test)
        
        # Metrics
        test_acc = accuracy_score(y_test, y_pred)
        test_f1 = f1_score(y_test, y_pred)
        test_precision = precision_score(y_test, y_pred)
        test_recall = recall_score(y_test, y_pred)
        
        # Try to get probabilities for AUC
        try:
            if hasattr(model, 'predict_proba'):
                y_proba = model.predict_proba(X_test)[:, 1]
                test_auc = roc_auc_score(y_test, y_proba)
            else:
                test_auc = np.nan
        except:
            test_auc = np.nan
        
        # Store results
        all_results.append({
            'Feature Set': feature_set.upper(),
            'Model': model_name,
            'Type': 'Individual',
            'Test Accuracy': test_acc,
            'Test F1': test_f1,
            'Test Precision': test_precision,
            'Test Recall': test_recall,
            'Test AUC': test_auc
        })
    
    # Evaluate ensemble models
    if feature_set in ensemble_models:
        for ensemble_name, ensemble_model in ensemble_models[feature_set].items():
            print(f'  Evaluating {ensemble_name}...')
            
            y_pred = ensemble_model.predict(X_test)
            
            test_acc = accuracy_score(y_test, y_pred)
            test_f1 = f1_score(y_test, y_pred)
            test_precision = precision_score(y_test, y_pred)
            test_recall = recall_score(y_test, y_pred)
            
            try:
                if hasattr(ensemble_model, 'predict_proba'):
                    y_proba = ensemble_model.predict_proba(X_test)[:, 1]
                    test_auc = roc_auc_score(y_test, y_proba)
                else:
                    test_auc = np.nan
            except:
                test_auc = np.nan
            
            all_results.append({
                'Feature Set': feature_set.upper(),
                'Model': ensemble_name,
                'Type': 'Ensemble',
                'Test Accuracy': test_acc,
                'Test F1': test_f1,
                'Test Precision': test_precision,
                'Test Recall': test_recall,
                'Test AUC': test_auc
            })

# Create DataFrame
df_all_results = pd.DataFrame(all_results)

print('\n' + '='*70)
print('COMPREHENSIVE RESULTS:')
print('='*70)
print(df_all_results.to_string(index=False))

# Save results
results_csv = RESULTS_DIR / '06a_comprehensive_results.csv'
df_all_results.to_csv(results_csv, index=False)
print(f'\n✓ Saved: {results_csv.name}')


EVALUATING ALL MODELS...

Feature set: MFCC
  Evaluating XGBoost...
  Evaluating Random Forest...
  Evaluating LinearSVC...
  Evaluating Stacking...
  Evaluating Soft Voting...
  Evaluating Hard Voting...

Feature set: GTCC
  Evaluating XGBoost...
  Evaluating Random Forest...
  Evaluating LinearSVC...
  Evaluating Stacking...
  Evaluating Soft Voting...
  Evaluating Hard Voting...

Feature set: COMBINED
  Evaluating XGBoost...
  Evaluating Random Forest...
  Evaluating LinearSVC...
  Evaluating Stacking...
  Evaluating Soft Voting...
  Evaluating Hard Voting...

COMPREHENSIVE RESULTS:
Feature Set         Model       Type  Test Accuracy  Test F1  Test Precision  Test Recall  Test AUC
       MFCC       XGBoost Individual       0.983210 0.984766        0.975382     0.994333  0.998479
       MFCC Random Forest Individual       0.974925 0.977298        0.965994     0.988869  0.994934
       MFCC     LinearSVC Individual       0.924003 0.930350        0.930727     0.929974  0.976663
      

### 4.2 Best Model Per Feature Set

In [ ]:
print('\nBEST MODEL PER FEATURE SET:')
print('='*70)

best_per_feature_set = []

for feature_set in FEATURE_SETS:
    # Filter results for this feature set
    fs_results = df_all_results[df_all_results['Feature Set'] == feature_set.upper()]
    
    # Find best by F1 score
    best_idx = fs_results['Test F1'].idxmax()
    best_row = fs_results.loc[best_idx]
    
    best_per_feature_set.append(best_row)
    
    print(f'\n{feature_set.upper()}:')
    print(f'  Best Model: {best_row["Model"]} ({best_row["Type"]})')
    print(f'  Test F1:    {best_row["Test F1"]:.4f}')
    print(f'  Test Acc:   {best_row["Test Accuracy"]:.4f}')
    print(f'  Precision:  {best_row["Test Precision"]:.4f}')
    print(f'  Recall:     {best_row["Test Recall"]:.4f}')

# Overall best
overall_best_idx = df_all_results['Test F1'].idxmax()
overall_best = df_all_results.loc[overall_best_idx]

print('\n' + '='*70)
print('OVERALL BEST MODEL:')
print('='*70)
print(f'Feature Set: {overall_best["Feature Set"]}')
print(f'Model:       {overall_best["Model"]} ({overall_best["Type"]})')
print(f'Test F1:     {overall_best["Test F1"]:.4f}')
print(f'Test Acc:    {overall_best["Test Accuracy"]:.4f}')
print(f'Precision:   {overall_best["Test Precision"]:.4f}')
print(f'Recall:      {overall_best["Test Recall"]:.4f}')
print('='*70)


BEST MODEL PER FEATURE SET:

MFCC:
  Best Model: Stacking (Ensemble)
  Test F1:    0.9856
  Test Acc:   0.9842
  Precision:  0.9788
  Recall:     0.9925

GTCC:
  Best Model: XGBoost (Individual)
  Test F1:    0.9829
  Test Acc:   0.9811
  Precision:  0.9727
  Recall:     0.9933

COMBINED:
  Best Model: XGBoost (Individual)
  Test F1:    0.9877
  Test Acc:   0.9864
  Precision:  0.9799
  Recall:     0.9955

OVERALL BEST MODEL:
Feature Set: COMBINED
Model:       XGBoost (Individual)
Test F1:     0.9877
Test Acc:    0.9864
Precision:   0.9799
Recall:      0.9955


### 4.3 Ensemble vs Individual Comparison

In [ ]:
print('\nENSEMBLE IMPROVEMENT ANALYSIS:')
print('='*70)

for feature_set in FEATURE_SETS:
    print(f'\n{feature_set.upper()}:')
    
    # Get results for this feature set
    fs_results = df_all_results[df_all_results['Feature Set'] == feature_set.upper()]
    
    # Best individual model
    individual_results = fs_results[fs_results['Type'] == 'Individual']
    if len(individual_results) > 0:
        best_individual = individual_results.loc[individual_results['Test F1'].idxmax()]
        print(f'  Best Individual: {best_individual["Model"]} (F1: {best_individual["Test F1"]:.4f})')
    
    # Ensemble models
    ensemble_results = fs_results[fs_results['Type'] == 'Ensemble']
    
    if len(ensemble_results) > 0:
        for _, ensemble_row in ensemble_results.iterrows():
            improvement = ensemble_row['Test F1'] - best_individual['Test F1']
            improvement_pct = (improvement / best_individual['Test F1']) * 100
            
            symbol = '✓' if improvement > 0 else '⚠️'
            print(f'  {symbol} {ensemble_row["Model"]}: F1 {ensemble_row["Test F1"]:.4f} '
                  f'({improvement:+.4f}, {improvement_pct:+.2f}%)')

print('\n' + '='*70)


ENSEMBLE IMPROVEMENT ANALYSIS:

MFCC:
  Best Individual: XGBoost (F1: 0.9848)
  ✓ Stacking: F1 0.9856 (+0.0009, +0.09%)
  ⚠️ Soft Voting: F1 0.8433 (-0.1415, -14.37%)
  ⚠️ Hard Voting: F1 0.9800 (-0.0048, -0.49%)

GTCC:
  Best Individual: XGBoost (F1: 0.9829)
  ⚠️ Stacking: F1 0.9826 (-0.0002, -0.03%)
  ⚠️ Soft Voting: F1 0.8294 (-0.1535, -15.62%)
  ⚠️ Hard Voting: F1 0.9780 (-0.0049, -0.50%)

COMBINED:
  Best Individual: XGBoost (F1: 0.9877)
  ⚠️ Stacking: F1 0.9875 (-0.0001, -0.01%)
  ⚠️ Soft Voting: F1 0.8581 (-0.1295, -13.11%)
  ⚠️ Hard Voting: F1 0.9830 (-0.0047, -0.47%)



## Section 5: Error Analysis

### 5.1 Confusion Matrices for Top Models

In [ ]:
print('\nCONFUSION MATRIX ANALYSIS:')
print('='*70)

# Get top 3 models overall
top_3_models = df_all_results.nlargest(3, 'Test F1')

confusion_matrices = {}

for idx, row in top_3_models.iterrows():
    feature_set = row['Feature Set'].lower()
    model_name = row['Model']
    model_type = row['Type']
    
    print(f'\n{feature_set.upper()} - {model_name} ({model_type}):')
    
    # Get model
    if model_type == 'Individual':
        model = loaded_models[feature_set][model_name]
    else:  # Ensemble
        model = ensemble_models[feature_set][model_name]
    
    # Get predictions
    X_test = test_data[feature_set]['X']
    y_test = test_data[feature_set]['y']
    y_pred = model.predict(X_test)
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    confusion_matrices[f'{feature_set}_{model_name}'] = cm
    
    # Calculate rates
    tn, fp, fn, tp = cm.ravel()
    
    fpr = fp / (fp + tn)  # False Positive Rate
    fnr = fn / (fn + tp)  # False Negative Rate
    
    print(f'  Confusion Matrix:')
    print(f'    TN: {tn:5d}  |  FP: {fp:5d}  (FPR: {fpr:.2%})')
    print(f'    FN: {fn:5d}  |  TP: {tp:5d}  (FNR: {fnr:.2%})')
    print(f'\n  False Positives: {fp:,} false alarms')
    print(f'  False Negatives: {fn:,} missed grinder detections (CRITICAL!)')
    print(f'\n  Safety: {(1-fnr)*100:.1f}% of grinder events detected')

print('\n' + '='*70)


CONFUSION MATRIX ANALYSIS:

COMBINED - XGBoost (Individual):
  Confusion Matrix:
    TN:  4011  |  FP:   101  (FPR: 2.46%)
    FN:    22  |  TP:  4919  (FNR: 0.45%)

  False Positives: 101 false alarms
  False Negatives: 22 missed grinder detections (CRITICAL!)

  Safety: 99.6% of grinder events detected

COMBINED - Stacking (Ensemble):
  Confusion Matrix:
    TN:  4020  |  FP:    92  (FPR: 2.24%)
    FN:    32  |  TP:  4909  (FNR: 0.65%)

  False Positives: 92 false alarms
  False Negatives: 32 missed grinder detections (CRITICAL!)

  Safety: 99.4% of grinder events detected

MFCC - Stacking (Ensemble):
  Confusion Matrix:
    TN:  4006  |  FP:   106  (FPR: 2.58%)
    FN:    37  |  TP:  4904  (FNR: 0.75%)

  False Positives: 106 false alarms
  False Negatives: 37 missed grinder detections (CRITICAL!)

  Safety: 99.3% of grinder events detected



### 5.2 Detailed Classification Reports

In [ ]:
print('\nDETAILED CLASSIFICATION REPORTS:')
print('='*70)

for idx, row in top_3_models.iterrows():
    feature_set = row['Feature Set'].lower()
    model_name = row['Model']
    model_type = row['Type']
    
    print(f'\n{feature_set.upper()} - {model_name} ({model_type}):')
    print('-'*70)
    
    # Get model and predictions
    if model_type == 'Individual':
        model = loaded_models[feature_set][model_name]
    else:
        model = ensemble_models[feature_set][model_name]
    
    X_test = test_data[feature_set]['X']
    y_test = test_data[feature_set]['y']
    y_pred = model.predict(X_test)
    
    # Classification report
    report = classification_report(
        y_test, y_pred,
        target_names=['Non-grinder', 'Grinder'],
        digits=4
    )
    print(report)

print('='*70)


DETAILED CLASSIFICATION REPORTS:

COMBINED - XGBoost (Individual):
----------------------------------------------------------------------
              precision    recall  f1-score   support

 Non-grinder     0.9945    0.9754    0.9849      4112
     Grinder     0.9799    0.9955    0.9877      4941

    accuracy                         0.9864      9053
   macro avg     0.9872    0.9855    0.9863      9053
weighted avg     0.9865    0.9864    0.9864      9053


COMBINED - Stacking (Ensemble):
----------------------------------------------------------------------
              precision    recall  f1-score   support

 Non-grinder     0.9921    0.9776    0.9848      4112
     Grinder     0.9816    0.9935    0.9875      4941

    accuracy                         0.9863      9053
   macro avg     0.9869    0.9856    0.9862      9053
weighted avg     0.9864    0.9863    0.9863      9053


MFCC - Stacking (Ensemble):
----------------------------------------------------------------------
   

### 5.3 Model Agreement Analysis

In [ ]:
print('\nMODEL AGREEMENT ANALYSIS:')
print('='*70)

# For each feature set, check where models agree/disagree
for feature_set in FEATURE_SETS:
    if feature_set not in loaded_models or len(loaded_models[feature_set]) < 2:
        continue
    
    print(f'\n{feature_set.upper()}:')
    
    X_test = test_data[feature_set]['X']
    y_test = test_data[feature_set]['y']
    
    # Get predictions from all models
    predictions = {}
    for model_name, model in loaded_models[feature_set].items():
        predictions[model_name] = model.predict(X_test)
    
    # Find samples where models disagree
    model_names = list(predictions.keys())
    if len(model_names) >= 2:
        pred1 = predictions[model_names[0]]
        pred2 = predictions[model_names[1]]
        
        disagreement_mask = pred1 != pred2
        n_disagreements = disagreement_mask.sum()
        
        print(f'  Models: {model_names[0]} vs {model_names[1]}')
        print(f'  Disagreements: {n_disagreements} / {len(y_test)} ({n_disagreements/len(y_test)*100:.1f}%)')
        
        # Where one is correct and other is wrong
        model1_correct = (pred1 == y_test) & disagreement_mask
        model2_correct = (pred2 == y_test) & disagreement_mask
        
        print(f'  {model_names[0]} correct, {model_names[1]} wrong: {model1_correct.sum()}')
        print(f'  {model_names[1]} correct, {model_names[0]} wrong: {model2_correct.sum()}')
        print(f'  Both wrong: {((pred1 != y_test) & (pred2 != y_test)).sum()}')

print('\n' + '='*70)


MODEL AGREEMENT ANALYSIS:

MFCC:
  Models: XGBoost vs Random Forest
  Disagreements: 123 / 9053 (1.4%)
  XGBoost correct, Random Forest wrong: 99
  Random Forest correct, XGBoost wrong: 24
  Both wrong: 128

GTCC:
  Models: XGBoost vs Random Forest
  Disagreements: 165 / 9053 (1.8%)
  XGBoost correct, Random Forest wrong: 138
  Random Forest correct, XGBoost wrong: 27
  Both wrong: 144

COMBINED:
  Models: XGBoost vs Random Forest
  Disagreements: 104 / 9053 (1.1%)
  XGBoost correct, Random Forest wrong: 86
  Random Forest correct, XGBoost wrong: 18
  Both wrong: 105



## Section 6: Production Metrics

### 6.1 Model Size Analysis

In [ ]:
print('\nMODEL SIZE ANALYSIS:')
print('='*70)

model_sizes = []

# Check all saved models
for model_file in MODELS_DIR.glob('*.pkl'):
    size_bytes = model_file.stat().st_size
    size_mb = size_bytes / (1024 * 1024)
    
    model_sizes.append({
        'Model File': model_file.name,
        'Size (MB)': size_mb,
        'Size (KB)': size_bytes / 1024
    })

df_sizes = pd.DataFrame(model_sizes).sort_values('Size (MB)', ascending=False)

print('\nTop 10 Largest Models:')
print(df_sizes.head(10).to_string(index=False))

print(f'\nDeployment Target: <50MB')
oversized = df_sizes[df_sizes['Size (MB)'] > 50]
if len(oversized) > 0:
    print(f'⚠️  {len(oversized)} models exceed 50MB target')
else:
    print('✓ All models within 50MB target')

# Save sizes
sizes_csv = RESULTS_DIR / '06a_model_sizes.csv'
df_sizes.to_csv(sizes_csv, index=False)
print(f'\n✓ Saved: {sizes_csv.name}')


MODEL SIZE ANALYSIS:

Top 10 Largest Models:
                  Model File  Size (MB)    Size (KB)
      knn_tuned_combined.pkl  84.163868 86183.800781
          knn_tuned_gtcc.pkl  67.395741 69013.238281
          knn_tuned_mfcc.pkl  67.395741 69013.238281
        voting_soft_gtcc.pkl  40.488907 41460.640625
           stacking_gtcc.pkl  39.931765 40890.126953
        voting_hard_gtcc.pkl  39.925974 40884.197266
random_forest_tuned_gtcc.pkl  38.859429 39792.055664
        voting_soft_mfcc.pkl  24.631622 25222.781250
           stacking_mfcc.pkl  24.164690 24744.642578
        voting_hard_mfcc.pkl  24.158915 24738.728516

Deployment Target: <50MB
⚠️  3 models exceed 50MB target

✓ Saved: 06a_model_sizes.csv


### 6.2 Inference Speed Benchmarking

In [ ]:
print('\nINFERENCE SPEED BENCHMARKING:')
print('='*70)
print('Target: <50ms per sample for real-time processing')

inference_times = []

# Benchmark top models
for idx, row in top_3_models.iterrows():
    feature_set = row['Feature Set'].lower()
    model_name = row['Model']
    model_type = row['Type']
    
    print(f'\n{feature_set.upper()} - {model_name} ({model_type}):')
    
    # Get model
    if model_type == 'Individual':
        model = loaded_models[feature_set][model_name]
    else:
        model = ensemble_models[feature_set][model_name]
    
    X_test = test_data[feature_set]['X']
    
    # Single sample inference (cold start)
    sample = X_test[0:1]
    start = time.time()
    _ = model.predict(sample)
    cold_start_time = (time.time() - start) * 1000  # ms
    
    # Warm inference (100 samples)
    n_samples = min(100, len(X_test))
    batch = X_test[:n_samples]
    
    start = time.time()
    _ = model.predict(batch)
    batch_time = time.time() - start
    per_sample_time = (batch_time / n_samples) * 1000  # ms
    
    # Throughput
    throughput = n_samples / batch_time  # samples/sec
    
    inference_times.append({
        'Feature Set': feature_set.upper(),
        'Model': model_name,
        'Type': model_type,
        'Cold Start (ms)': cold_start_time,
        'Per Sample (ms)': per_sample_time,
        'Throughput (samples/s)': throughput,
        'Batch Size': n_samples
    })
    
    status = '✓' if per_sample_time < 50 else '⚠️'
    print(f'  {status} Cold start:  {cold_start_time:.2f} ms')
    print(f'  {status} Per sample:  {per_sample_time:.2f} ms')
    print(f'     Throughput:   {throughput:.1f} samples/sec')

df_inference = pd.DataFrame(inference_times)

print('\n' + '='*70)
print('INFERENCE SPEED SUMMARY:')
print(df_inference.to_string(index=False))

# Save
inference_csv = RESULTS_DIR / '06a_inference_times.csv'
df_inference.to_csv(inference_csv, index=False)
print(f'\n✓ Saved: {inference_csv.name}')


INFERENCE SPEED BENCHMARKING:
Target: <50ms per sample for real-time processing

COMBINED - XGBoost (Individual):
  ✓ Cold start:  1.44 ms
  ✓ Per sample:  0.01 ms
     Throughput:   77643.5 samples/sec

COMBINED - Stacking (Ensemble):
  ✓ Cold start:  18.22 ms
  ✓ Per sample:  0.29 ms
     Throughput:   3482.1 samples/sec

MFCC - Stacking (Ensemble):
  ✓ Cold start:  31.56 ms
  ✓ Per sample:  0.43 ms
     Throughput:   2319.6 samples/sec

INFERENCE SPEED SUMMARY:
Feature Set    Model       Type  Cold Start (ms)  Per Sample (ms)  Throughput (samples/s)  Batch Size
   COMBINED  XGBoost Individual         1.444817         0.012879            77643.539430         100
   COMBINED Stacking   Ensemble        18.221855         0.287182             3482.108374         100
       MFCC Stacking   Ensemble        31.559944         0.431101             2319.640299         100

✓ Saved: 06a_inference_times.csv


### 6.3 Memory Footprint (Loaded Model)

In [ ]:
import sys

print('\nMEMORY FOOTPRINT ANALYSIS:')
print('='*70)

memory_usage = []

for idx, row in top_3_models.iterrows():
    feature_set = row['Feature Set'].lower()
    model_name = row['Model']
    model_type = row['Type']
    
    # Get model
    if model_type == 'Individual':
        model = loaded_models[feature_set][model_name]
    else:
        model = ensemble_models[feature_set][model_name]
    
    # Estimate memory (approximate)
    mem_bytes = sys.getsizeof(model)
    mem_mb = mem_bytes / (1024 * 1024)
    
    memory_usage.append({
        'Feature Set': feature_set.upper(),
        'Model': model_name,
        'Type': model_type,
        'Memory (MB)': mem_mb,
        'Memory (KB)': mem_bytes / 1024
    })
    
    print(f'{feature_set.upper()} - {model_name}: {mem_mb:.2f} MB')

df_memory = pd.DataFrame(memory_usage)

print('\n' + '='*70)
print('Note: Memory estimates are approximate (sys.getsizeof)')
print('Actual runtime memory may be higher due to internal structures')

# ESP32-S3 has ~500KB SRAM
print('\n⚠️  ESP32-S3 SRAM: ~500KB (~0.5MB)')
print('Models will NOT fit in ESP32 memory - need quantization!')
print('This will be addressed in later compression notebooks.')


MEMORY FOOTPRINT ANALYSIS:
COMBINED - XGBoost: 0.00 MB
COMBINED - Stacking: 0.00 MB
MFCC - Stacking: 0.00 MB

Note: Memory estimates are approximate (sys.getsizeof)
Actual runtime memory may be higher due to internal structures

⚠️  ESP32-S3 SRAM: ~500KB (~0.5MB)
Models will NOT fit in ESP32 memory - need quantization!
This will be addressed in later compression notebooks.


## Section 7: Visualization

In [ ]:
print('\nGenerating visualizations...')

# Figure 1: Model Performance Comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy comparison
for feature_set in FEATURE_SETS:
    fs_data = df_all_results[df_all_results['Feature Set'] == feature_set.upper()]
    axes[0].plot(fs_data.index, fs_data['Test Accuracy'], 
                 marker='o', label=feature_set.upper(), linewidth=2)

axes[0].set_xlabel('Model Index', fontsize=12)
axes[0].set_ylabel('Test Accuracy', fontsize=12)
axes[0].set_title('Test Accuracy Across Models', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# F1 comparison
for feature_set in FEATURE_SETS:
    fs_data = df_all_results[df_all_results['Feature Set'] == feature_set.upper()]
    axes[1].plot(fs_data.index, fs_data['Test F1'], 
                 marker='o', label=feature_set.upper(), linewidth=2)

axes[1].set_xlabel('Model Index', fontsize=12)
axes[1].set_ylabel('Test F1 Score', fontsize=12)
axes[1].set_title('Test F1 Across Models', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
fig1_path = FIGURES_DIR / '06a_model_comparison.png'
plt.savefig(fig1_path, dpi=300, bbox_inches='tight')
print(f'✓ Saved: {fig1_path.name}')
plt.close()

# Figure 2: Confusion matrices for top 3
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax_idx, (idx, row) in enumerate(top_3_models.iterrows()):
    if ax_idx >= 3:
        break
    
    feature_set = row['Feature Set'].lower()
    model_name = row['Model']
    
    cm_key = f'{feature_set}_{model_name}'
    if cm_key in confusion_matrices:
        cm = confusion_matrices[cm_key]
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[ax_idx],
                    xticklabels=['Non-grinder', 'Grinder'],
                    yticklabels=['Non-grinder', 'Grinder'])
        axes[ax_idx].set_title(f'{feature_set.upper()} - {model_name}', fontweight='bold')
        axes[ax_idx].set_ylabel('Actual')
        axes[ax_idx].set_xlabel('Predicted')

plt.tight_layout()
fig2_path = FIGURES_DIR / '06a_confusion_matrices_top3.png'
plt.savefig(fig2_path, dpi=300, bbox_inches='tight')
print(f'✓ Saved: {fig2_path.name}')
plt.close()

print('\n✓ All visualizations complete')


Generating visualizations...
✓ Saved: 06a_model_comparison.png
✓ Saved: 06a_confusion_matrices_top3.png

✓ All visualizations complete


## Section 8: Final Model Selection & Recommendation

In [ ]:
print('\n' + '='*70)
print('FINAL MODEL SELECTION & RECOMMENDATION')
print('='*70)

# Multi-criteria scoring
print('\nCriteria Weighting:')
print('  Test F1:           40%')
print('  Recall (Safety):   30%')
print('  Inference Speed:   20%')
print('  Model Size:        10%')

# Create decision matrix
decision_data = []

for idx, row in top_3_models.iterrows():
    feature_set = row['Feature Set'].lower()
    model_name = row['Model']
    model_type = row['Type']
    
    # Get metrics
    f1_score_val = row['Test F1']
    recall = row['Test Recall']
    
    # Get inference time
    inf_row = df_inference[
        (df_inference['Feature Set'] == feature_set.upper()) &
        (df_inference['Model'] == model_name)
    ]
    if len(inf_row) > 0:
        inference_ms = inf_row['Per Sample (ms)'].values[0]
    else:
        inference_ms = 50  # Default
    
    # Get model size
    if model_type == 'Individual':
        if model_name == 'XGBoost':
            size_file = f'xgboost_tuned_{feature_set}.pkl'
        elif model_name == 'Random Forest':
            size_file = f'random_forest_tuned_{feature_set}.pkl'
        else:
            size_file = f'linearsvc_tuned_{feature_set}.pkl'
    else:
        if model_name == 'Stacking':
            size_file = f'stacking_{feature_set}.pkl'
        elif 'Soft' in model_name:
            size_file = f'voting_soft_{feature_set}.pkl'
        else:
            size_file = f'voting_hard_{feature_set}.pkl'
    
    size_row = df_sizes[df_sizes['Model File'] == size_file]
    if len(size_row) > 0:
        size_mb = size_row['Size (MB)'].values[0]
    else:
        size_mb = 10  # Default
    
    # Normalize and score
    # F1: higher is better (normalize to 0-1)
    f1_norm = f1_score_val  # Already 0-1
    
    # Recall: higher is better
    recall_norm = recall
    
    # Inference: lower is better (invert and normalize)
    speed_norm = max(0, 1 - (inference_ms / 100))  # 0ms = 1.0, 100ms+ = 0
    
    # Size: lower is better
    size_norm = max(0, 1 - (size_mb / 100))  # 0MB = 1.0, 100MB+ = 0
    
    # Weighted score
    score = (
        0.40 * f1_norm +
        0.30 * recall_norm +
        0.20 * speed_norm +
        0.10 * size_norm
    )
    
    decision_data.append({
        'Feature Set': feature_set.upper(),
        'Model': model_name,
        'Type': model_type,
        'F1': f1_score_val,
        'Recall': recall,
        'Inference (ms)': inference_ms,
        'Size (MB)': size_mb,
        'Score': score
    })

df_decision = pd.DataFrame(decision_data).sort_values('Score', ascending=False)

print('\nMULTI-CRITERIA RANKING:')
print(df_decision.to_string(index=False))

# Winner
winner = df_decision.iloc[0]
runner_up = df_decision.iloc[1] if len(df_decision) > 1 else None

print('\n' + '='*70)
print('RECOMMENDED PRIMARY MODEL:')
print('='*70)
print(f'Feature Set:  {winner["Feature Set"]}')
print(f'Model:        {winner["Model"]} ({winner["Type"]})')
print(f'Test F1:      {winner["F1"]:.4f}')
print(f'Recall:       {winner["Recall"]:.4f} ({(1-winner["Recall"])*100:.1f}% missed grinders)')
print(f'Inference:    {winner["Inference (ms)"]:.1f} ms/sample')
print(f'Size:         {winner["Size (MB)"]:.1f} MB')
print(f'Overall Score: {winner["Score"]:.3f}')

if runner_up is not None:
    print('\n' + '='*70)
    print('BACKUP MODEL:')
    print('='*70)
    print(f'Feature Set:  {runner_up["Feature Set"]}')
    print(f'Model:        {runner_up["Model"]} ({runner_up["Type"]})')
    print(f'Test F1:      {runner_up["F1"]:.4f}')
    print(f'Recall:       {runner_up["Recall"]:.4f}')
    print(f'Inference:    {runner_up["Inference (ms)"]:.1f} ms/sample')
    print(f'Size:         {runner_up["Size (MB)"]:.1f} MB')

print('\n' + '='*70)
print('NEXT STEPS:')
print('='*70)
print('1. Proceed to Notebook 06b for:')
print('   - SHAP interpretability analysis')
print('   - Feature reduction for ESP32-S3 deployment')
print('   - Identify minimal feature set (target: <100ms extraction)')
print('\n2. Current models are too large for ESP32-S3:')
print('   - Need quantization & compression (later notebook)')
print('   - Target: <500KB model size')
print('\n3. Feature extraction optimization critical:')
print(f'   - {winner["Feature Set"]}: need to reduce from 138+ features')
print('   - Target: 30-50 features for <100ms extraction on ESP32')
print('='*70)


FINAL MODEL SELECTION & RECOMMENDATION

Criteria Weighting:
  Test F1:           40%
  Recall (Safety):   30%
  Inference Speed:   20%
  Model Size:        10%

MULTI-CRITERIA RANKING:
Feature Set    Model       Type       F1   Recall  Inference (ms)  Size (MB)    Score
   COMBINED  XGBoost Individual 0.987652 0.995547        0.012879   1.573633 0.992126
   COMBINED Stacking   Ensemble 0.987528 0.993524        0.287182  16.967043 0.975527
       MFCC Stacking   Ensemble 0.985630 0.992512        0.431101  24.164690 0.966978

RECOMMENDED PRIMARY MODEL:
Feature Set:  COMBINED
Model:        XGBoost (Individual)
Test F1:      0.9877
Recall:       0.9955 (0.4% missed grinders)
Inference:    0.0 ms/sample
Size:         1.6 MB
Overall Score: 0.992

BACKUP MODEL:
Feature Set:  COMBINED
Model:        Stacking (Ensemble)
Test F1:      0.9875
Recall:       0.9935
Inference:    0.3 ms/sample
Size:         17.0 MB

NEXT STEPS:
1. Proceed to Notebook 06b for:
   - SHAP interpretability analysis
   -

## Section 9: Save Final Models & Documentation

In [ ]:
from datetime import datetime

print('\nSaving final documentation...')

# Save decision matrix
decision_csv = RESULTS_DIR / '06a_final_model_selection.csv'
df_decision.to_csv(decision_csv, index=False)
print(f'✓ Saved: {decision_csv.name}')

# Create recommendation report
report = {
    'timestamp': datetime.now().isoformat(),
    'primary_model': {
        'feature_set': winner['Feature Set'],
        'model_name': winner['Model'],
        'model_type': winner['Type'],
        'test_f1': float(winner['F1']),
        'recall': float(winner['Recall']),
        'inference_ms': float(winner['Inference (ms)']),
        'size_mb': float(winner['Size (MB)']),
        'overall_score': float(winner['Score'])
    },
    'total_models_evaluated': len(df_all_results),
    'ensemble_models_built': len(df_all_results[df_all_results['Type'] == 'Ensemble']),
    'deployment_status': 'Needs feature reduction and quantization for ESP32-S3'
}

if runner_up is not None:
    report['backup_model'] = {
        'feature_set': runner_up['Feature Set'],
        'model_name': runner_up['Model'],
        'test_f1': float(runner_up['F1'])
    }

report_json = RESULTS_DIR / '06a_recommendation_report.json'
with open(report_json, 'w') as f:
    json.dump(report, f, indent=2)
print(f'✓ Saved: {report_json.name}')

print('\n' + '='*70)
print('✓ NOTEBOOK 06a COMPLETE')
print('='*70)
print(f'\nOutputs:')
print(f'  Results:  {RESULTS_DIR.name}/')
print(f'  Figures:  {FIGURES_DIR.name}/')
print(f'  Models:   {MODELS_DIR.name}/')
print(f'\nReady for Notebook 06b: Feature Importance & Deployment Optimization')
print('='*70)


Saving final documentation...
✓ Saved: 06a_final_model_selection.csv
✓ Saved: 06a_recommendation_report.json

✓ NOTEBOOK 06a COMPLETE

Outputs:
  Results:  results/
  Figures:  figures/
  Models:   classical/

Ready for Notebook 06b: Feature Importance & Deployment Optimization
